In [ ]:
from pathlib import Path
import os

from IPython.display import display
from pathlib import Path
from rasterio import features
import os
import pandas as pd
from satellites.shared.constants import SEED_NUMBER
from satellites.shared.notebook import configure_notebook
configure_notebook(seed=42, tensorflow=True)
import satellites.shared.io as io_l

from satellites.ml_classification import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

In [ ]:
WORK_PATH = Path("C:/work_dir/neuro_2024-2025")

PARCELS_PATH = WORK_PATH / "1_eo_parcel_stats_fill" / "satellite_parcel_time_stats.geoparquet"
EDA_REPORT_PATH = WORK_PATH / "2_eo_results_eda_fill" / "eda_annotated_data.parquet"
PROPOSED_FEATURES_PATH = WORK_PATH / "2_eo_results_eda_fill" / "tables" / "feature_selection" / "feature_selection_proposal.csv"
OUTPUT_PATH = WORK_PATH / "3_eo_ml_classification_fill"

class_column = "crop_category"
parcel_id_column = "parcel_id"


In [ ]:
gdf_parcels = io_l.read_data(str(PARCELS_PATH))
eda_parcels = io_l.read_data(str(EDA_REPORT_PATH))
eda_parcels = eda_parcels[[parcel_id_column, "is_outlier_97_5pct"]]
gdf_parcels = gdf_parcels.merge(eda_parcels, on=parcel_id_column, how="left")
gdf_parcels.head(3)

In [ ]:
gdf_parcels = gdf_parcels[gdf_parcels[class_column] != "jjjj"]
label_counts = gdf_parcels[class_column].value_counts()
print(f"Number of unique labels: {len(label_counts)}")
print(gdf_parcels[class_column].value_counts())
print(50*"-")
rare_labels = label_counts[label_counts < 5].index
gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = "9999"
print(f"Number of unique labels after grouping rare labels: {len(gdf_parcels[class_column].value_counts())}")
print(gdf_parcels[class_column].value_counts())

In [ ]:
df_features = io_l.read_data(str(PROPOSED_FEATURES_PATH))
display(df_features.head())

proposed_features = df_features[df_features["proposed_action"]=="use"]["feature"].tolist()
print(f"Proposed features: {len(proposed_features)}")
proposed_features

In [ ]:
gdf_parcels = gdf_parcels.to_crs(2100)
gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

base_cols = [parcel_id_column, class_column, "geometry"]
extra_features = ["elevation_mean_m", "GAIOIKANOT", "x", "y"]
temporal_features = ("_median", "_sd", "_range",)
selected_indices = ("NDMI", "NDRE", "SAVI", "NDVI", "EVI", "NDWI",)
original_features = [x for x in gdf_parcels.columns if any(keyword in x for keyword in temporal_features) and any(idx in x for idx in selected_indices)]

gdf_parcels = gdf_parcels[base_cols + original_features + extra_features].copy()
features = original_features + extra_features

gdf_parcels = gdf_parcels[gdf_parcels[class_column] != 9999].copy()
print(f"Final features for modeling ({len(features)} features): {features}")
display(gdf_parcels.head(3))

In [ ]:
config = ClassificationPipelineConfig(
    project_dir=OUTPUT_PATH,
    target_column=class_column,
    feature_columns=features,
    id_column=parcel_id_column,

    apply_iqr=False,
    spatial_interpolation_method=None,

    cv_folds=5,
    test_size=0.2,
    spatial_split=True,
    spatial_split_grid_size=30,
    spatial_split_method="by_row",

    random_state=SEED_NUMBER,
    # n_jobs=1,

    scoring_primary="f1_macro",
    cv_ranking_method="score_minus_std",
    selection_type="soft_voting",
    top_voting_models=6,
    max_search_candidates=40,
    prediction_confidence_threshold=0.60,

    interpretability_top_models=5,
    interpretability_include_shap=False,

    optimize_class_probabilities=True,
    probability_multiplier_grid=(0.8, 1.0, 1.2, 1.5, 2.0),
    probability_optimization_iterations=2,
    probability_optimization_max_accuracy_drop=0.02,

    rank_confidence_enabled=True,
    force_retrain_models=False,
    class_reliability_enabled=False,

    label_balancing_method="random_oversample",
    lstm_temporal_statistics=temporal_features,
    selected_models=(
        "bagging",
        #"bayesian",
        "decision_tree",
        "extra_trees",
        "random_forest",
        "gradient_boosting",
        "lightgbm",
        "xgboost",
        # "knn",
        # "neural_network",
        # "tensorflow_neural_network",
         "keras_lstm",
   ),
    max_map_geometries=20000,
    open_html_report=True,
)

layer_features = max(16, min(len(features), 64))
config.tune_params["tensorflow_neural_network"]["model__hidden_layer_sizes"] = [(layer_features,), (layer_features, layer_features), (layer_features, layer_features // 2)]
config.tune_params["neural_network"]["model__hidden_layer_sizes"] = [(layer_features,), (layer_features, layer_features), (layer_features, layer_features // 2)]
config.tune_params["keras_lstm"]["model__lstm_units_1"] = [16, 32, 64]
pipeline = GeospatialClassificationPipeline(config)


In [ ]:
summary_check = pipeline.run_check(df=gdf_parcels)
summary_check

In [ ]:
summary_prepare = pipeline.run_prepare()
summary_prepare

In [ ]:
summary_train = pipeline.run_train()
summary_train

In [ ]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate

In [ ]:
summary_predict = pipeline.run_predict()
summary_predict